In [ ]:
####RAG Pipelines- Data Ingestion to Vector DB Pipeline

import os
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

import chromadb
from chromadb.config import Settings
from typing import List, Dict, Tuple, Any


import numpy as np
from sentence_transformers import SentenceTransformer




from pathlib import Path

In [ ]:

# Define embeddingmanager class to generate embeddings
class EmbeddingManager:
    def __init__(self, model_name):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """
        Load the embedding model based on the specified model name.
        """
        try :
            self.embedding_model = SentenceTransformer(self.model_name)
            print ("Successfully loaded model ", self.model_name)
            print ("Model dimension: ", self.embedding_model.get_embedding_dimension())

        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    
    def generate_embeddings(self, texts : List[str]) -> list[np.ndarray]:
        """
        Generate embeddings for a list of texts using the specified embedding model.
        
        Args:
            Texts : A list of String objects whose content will be converted into embeddings.
        
        Returns:
            A list of numpy arrays, where each array is the embedding of the corresponding text.
        """
        try:
            #texts = []
            #embeddings = []
        
            #texts = [doc.page_content for doc in Documents]
            embeddings = self.embedding_model.encode(texts,show_progress_bar=True)
            print ("Embeddings done ", embeddings.size)
            return embeddings
        except Exception as e:
            print(f"Error generating embeddings: {e}")
            raise





In [ ]:
class VectorStoreDB:
    def __init__(self, persist_directory: str, collection_name: str):
       
        self.persistent_directory =  persist_directory
        self.client = None
        self.collection = None
        self.collection_name = collection_name
        self.create_vector_store (self.persistent_directory,self.collection_name)

    def create_vector_store(self,persist_directory: str, collection_name: str):
        """
        Create a vector store with the specified persistence directory and collection name.
        """
        try :
            self.persistent_directory = os.path.abspath(persist_directory)



            self.client = chromadb.PersistentClient(path=self.persistent_directory,
                                                     settings=Settings(anonymized_telemetry=False))
            


            self.collection = self.client.get_or_create_collection(
                                name=collection_name,
                                metadata={"description": "Collection for storing CSV document embeddings"}
                                )
            print ("successfully created vector store with collection name ", collection_name)

        except Exception as e:
            print(f"Error creating vector store: {e}")
            raise


    def add_documents(self, documents: list[Document], embeddings: list[np.ndarray]):
        ###
        #Add or update documents and their corresponding embeddings in the vector store idempotently.
        ##
        if len(documents) != len(embeddings):
            raise ValueError("The number of documents must match the number of embeddings.")
        
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Safe extraction whether it's a Document object or a Dict
            if hasattr(doc, "metadata"):
                metadata = dict(doc.metadata)
                page_content = doc.page_content
            else:
                metadata = dict(doc.get("metadata", {}))
                page_content = doc.get("page_content", "")                  
            
            # --- 1. GENERATE DETERMINISTIC UNIQUE ID ---
            # Get the source filename or fall back to a default identifier
            source_file = metadata.get("source", "unknown_source")
            # Clean up the file path string to create a clean alphanumeric ID token
            clean_source = os.path.basename(source_file).replace('.', '_').replace(' ', '_')
            
            # This ID stays identical across script restarts for the exact same file chunks
            doc_id = f"doc_{clean_source}_{i}"  
            
            ids.append(doc_id)
            metadata["doc_index"] = i
            metadata["content_length"] = len(page_content)
            
            metadatas.append(metadata)
            documents_text.append(page_content)
            embeddings_list.append(embedding.tolist())  # Convert numpy array to list for storage
        
        # --- 2. USE UPSERT INSTEAD OF ADD ---
        # upsert() will insert if the ID is new, or update/overwrite if the ID already exists
        self.collection.upsert(
            ids=ids,
            metadatas=metadatas,
            documents=documents_text,
            embeddings=embeddings_list
        )
        print(f"Successfully indexed/upserted documents in collection '{self.collection_name}'. Current total count: {self.collection.count()}")






In [ ]:

### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
#Split the documents into chunks
# 
# 
def splitDocuments(docs):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    chunks = text_splitter.split_documents(all_pdf_documents)
    return chunks


all_pdf_documents = process_all_pdfs("../data")
chunks = splitDocuments(all_pdf_documents)
print ("Total chunks created ",len(chunks))
embedding_manager = EmbeddingManager(model_name="all-MiniLM-L6-v2")
# Extract the text strings from the LangChain document objects
texts_to_embed = [doc.page_content for doc in chunks]
# Pass the list of strings to your manager
embeddings = embedding_manager.generate_embeddings(texts_to_embed)

# Store into vector db
vectorstoredb = VectorStoreDB(persist_directory="../data/vectorstore",collection_name="pdf_collection")
print ("Vector store created successfully with collection name ", vectorstoredb.collection.name)
vectorstoredb.add_documents(chunks, embeddings)  # Add documents and embeddings to the vector store
print ("Completed all tasks")


In [ ]:
class RAGRetriever:

    """ Handles query based retrieval
    """

    def __init__(self, vector_store: VectorStoreDB, embedding_manager : EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve (self, query: str, top_k : int = 5, score_threshold: float = 0.0) -> List[Dict[str,Any]]:
        # Retrieve relevant documents for a query
        #
        # Args: 
        #       query - The search string
        #       top_k - number of top results to return
        #       score_threshold - Minimum similarity score threshold
        # Returns :
        #       List of retrieved documents and metadata

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k} and Score Threshold: {score_threshold}")

        # Generating embeddings
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
       # query_embedding = self.embedding_model.encode(query, convert_to_numpy=True)
        #print (f"Query embedding done : {query_embedding}")

        # search in the vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings = [query_embedding.tolist()],
                n_results = top_k
            )

            retrieved_docs = []

            if (results['documents'] and results['documents'][0] ):
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]


                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    similarity_score = 1 - distance
                    if (similarity_score >= score_threshold):
                        retrieved_docs.append({
                                'id': doc_id,
                                'document': document,
                                'metadata': metadata,
                                'similarity_score': similarity_score,
                                'distance': distance,
                                'rank' : i+1
                        })
                print (f"Retrieved {len(retrieved_docs)} documents after filtering")
            else:
                print ("No documents found")

            return retrieved_docs

        except Exception as e:
            print (f"Error during retrieval {e}")
            return []


ragRetriever = RAGRetriever(vectorstoredb,embedding_manager)
#ragRetriver.retrieve("Unified Multi-task Learning Framework")

In [ ]:
# Simple RAG pipeline with LLM Integration
from langchain_groq import ChatGroq
from dotenv import load_dotenv

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")


llm = ChatGroq(groq_api_key=groq_api_key, model_name="openai/gpt-oss-120b",temperature=0.1,max_tokens=1024)

query = "What is attention mechanism"

# 2 Simple RAG function
# Retrieve Context + generate response

def rag_simple(query, retriever, llm,top_k=3):
    results = retriever.retrieve(query,top_k=top_k)
    
    context = "\n\n".join([doc.get('document', doc.get('text', '')) for doc in results]) if results else ""

    if not context:
        return "No relevant context found to the query"

    ### generate the answer using groq llm
    prompt = """Use the following context to answer the question concisely
            Context :
            {context}

            Query:
            {query}
            """

    response = llm.invoke([prompt.format(context=context,query=query)])
    return response.content

answer= rag_simple(query, ragRetriver, llm )
answer


In [ ]:
### Enhanced RAG Pipeline
from langchain_groq import ChatGroq
from dotenv import load_dotenv

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")


llm = ChatGroq(groq_api_key=groq_api_key, model_name="openai/gpt-oss-120b",temperature=0.1,max_tokens=1024)

def rag_advanced (query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    ## Advanced RAG Retriever
     #   additional features
     #   will add sources, confidence score and context if needed
   

    results = retriever.retrieve (query,top_k=top_k,score_threshold=min_score)
    # Results should be a list of dictionaries
    
    if not results:
        return {'answer':"No relevant context found",
                'sources':[],
                'confidence': 0,
                'context': ""
                }

   
    # 1. Safely extract context
    context = "\n\n".join([doc.get('document', doc.get('text', '')) for doc in results]) if results else ""
    # 2. Safely build sources (using .get() with a default fallback string for preview)
    sources=[]
    sources.extend({
        'source': doc.get('metadata', {}).get('source_file', doc.get('metadata', {}).get('source', 'unknown')),
        'page': doc.get('metadata', {}).get('page', 'unknown'),
        'score': doc.get('similarity_score', 'unknown'),
        'preview': (doc.get('document') or doc.get('text') or "")[:120] + "..."
    } for doc in results if doc is not None)
            
    # 3. Safely calculate max confidence (filtering out None)
    confidence = max((doc.get('similarity_score', 0) for doc in results if doc is not None), default=0)
    
    prompt = """Use the following context to answer the question concisely
                Context :
                {context}
    
                Query:
                {query}
                """
    print (f"*** CONTEXT ***\n{context}\n*****")
    response = llm.invoke([prompt.format(context=context,query=query)])
    output = {
            'answer': response.content,
            'sources': sources,
            'confidence': confidence,
    }
    if (return_context):
        output['context'] = context
        
    return output

query = "What are hard negative mining techniques"
answer = rag_advanced (query, retriever= ragRetriever, llm=llm,top_k=5, min_score=0.1,return_context=True )
answer

    


In [ ]:
# Advanced RAG Pipeline: Streaming, Citations, History, Summarization
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:

        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found"
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc.get('document', doc.get('text', '')) for doc in results])
            sources=[]
            sources.extend({
                    'source': doc.get('metadata', {}).get('source_file', doc.get('metadata', {}).get('source', 'unknown')),
                    'page': doc.get('metadata', {}).get('page', 'unknown'),
                    'score': doc.get('similarity_score', 'unknown'),
                    'preview': (doc.get('document') or doc.get('text') or "")[:120] + "..."
            } for doc in results if doc is not None)
            prompt = """Use the following context to answer the question concisely
                            Context :
                            {context}
                
                            Query:
                            {query}
                        """
            #Streaming Answer simulation

            if (stream):
                print ("Streaming answer")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
            print()
            response = self.llm.invoke([prompt.format(context=context, query=question)])
            answer = response.content
        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n\n"+"\n\n".join (citations) if citations else answer
        # Optionally summarize answer
        summary = None
        if (summarize and answer):
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content
        self.history.append({
                    "question": question,
                    "answer": answer,
                    "summary": summary,
                    "sources": sources
        })
        return {
                    "question": question,
                    "answer": answer,
                    "summary": summary,
                    "sources": sources,
                    "history": self.history
        }


# Example usage
adv_rag = AdvancedRAGPipeline(ragRetriever,llm)
result = adv_rag.query("what is attention is all you need",top_k=3, min_score=0.1, stream=True, summarize=True)
print ("Final Answer\n")
print (f"Question: {result['question']}\n")
print (f"Answer: {result['answer']}\n")
print (f"Summary: {result['summary']}\n")
print (f"History: {result['history']}\n")

            
